# betacomp.ipynb

Input .spec files, output DataFrame with beta, dlogbeta, dlogbeta_corr computed

If generate_synthetic is True, generate synthetic spectra to compute the above values from.

In [7]:
# %matplotlib ipympl


import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import struct
import math
import time
import sys
import string

sys.path.append('..')

import stressdrop_file_IO as sdio
import seismo_functions as sf
import mapping_tools as mt
import importlib
importlib.reload(sdio)
import obspy

from scipy import sparse
import utm
import copy 

import ipywidgets as widgets
from ipywidgets import interactive

from shapely.geometry import Point, MultiPoint
from shapely.geometry.polygon import Polygon

from lib.beta_functions import read_beta_files, read_betatxt_files, read_spec_df, read_spec_max_df

from tqdm import trange


In [8]:
### PATHS AND DIRECTORIES ###
# Location of .spec files
# results from Vandevert et al. 2024
# p_spec_dir = "/Users/ivandevert/projects/ridgecrest2019_prev/proc/spectral/01_compspec/SPECOUT/"
# s_spec_dir = "/Users/ivandevert/projects/ridgecrest2019_prev/proc/spectral_s/01_compspec/SPECOUT/"

# Multitaper spectra
p_spec_dir = "/Users/ivandevert/projects/ridgecrest2019/proc/specdecomp_p/01_compspec/SPECOUT/"
s_spec_dir = "/Users/ivandevert/projects/ridgecrest2019/proc/specdecomp_s/01_compspec/SPECOUT/"

# Location of earthquake catalog containing stress drop estimates
delsig_p_catalog_path = "/Users/ivandevert/projects/ridgecrest2019_prev/comparisons/p_s_comparison/final/ridgecrest_stressdrop_p.txt"
delsig_s_catalog_path = "/Users/ivandevert/projects/ridgecrest2019_prev/comparisons/p_s_comparison/final/ridgecrest_stressdrop_s.txt"

# Location of spectra.pkl file/where to save it
spectra_dir = "/Users/ivandevert/projects/spectral_falloff_ratio/data/spectra/"

paper_figure_dir = "/Users/ivandevert/Documents/papers/high_freq_ratio/paper_name_pre_submission/figs/"


### INPUT DATA PARAMETERS ###
# phase = 'p' 
phase = 's' 
units = ['h', 'n']

# Beta computation params
stn_f_range = [2.5, 6.0]
dist_min = 0
dist_max = 100

stn_req = 5.0

### P-WAVE PARAMETERS ###
low_beta_window_p = [1, 5]
high_beta_window_p = [15, 22]
calib_mag_range_p = [1.4, 1.6]
fc_fixed_p = 30

### S-WAVE PARAMETERS ###
low_beta_window_s = [1, 5]
high_beta_window_s = [15, 22]
calib_mag_range_s = [1.9, 2.1]
fc_fixed_s = 17

# calibration event parameters
calib_rmax = 10.0
calib_zmax = 2.0
calib_stn_req = 5.0
ncalib_min = 10

# miscellaneous parameters
xsec_dmax = 1500
f_nyquist = 50.0
magnitude_correction = "smoothedspline" # "simple"
mag_corr_dM = 0.1

# SYNTHETIC OPTIONS
generate_synthetic = False
syn_dataset_name = "SYN3"
add_spectrum_noise = True



# process some of the parameters
units = [el.upper() for el in units]

if phase == 'p':
    components = ['Z']
    calib_mag_range = calib_mag_range_p
    low_beta_window = low_beta_window_p
    high_beta_window = high_beta_window_p
    spec_dir = p_spec_dir
    fc_fixed = fc_fixed_p
elif phase == 's':
    components = ['N', 'E', '1', '2']

    calib_mag_range = calib_mag_range_s
    low_beta_window = low_beta_window_s
    high_beta_window = high_beta_window_s
    spec_dir = s_spec_dir
    fc_fixed = fc_fixed_s





In [9]:
if generate_synthetic:

    # synthetic data parameters
    syn_nevents = 100000
    syn_nstations = 50
    syn_delsig_fixed = 5.0E6
    syn_qlat_range = [35.5, 36.0]
    syn_qlon_range = [-117.75, -117.34]
    syn_qdep_range = [0, 20.0]
    syn_qmag_range = [1, 7.1]
    syn_slat_range = [35.5, 36.0]
    syn_slon_range = [-117.75, -117.34]
    syn_selev_range = [0, 500]
    syn_nts_range = [15, 20]
    syn_k = 0.38
    syn_Vs = 3464.0
    syn_nf = 76
    syn_f = np.linspace(0, f_nyquist, syn_nf)
    logomega0_corr = 19.0

    # set the random seed
    rng = np.random.default_rng(42)

    # generate event-specific data using the above ranges
    syn_event_id = np.arange(1000000, 1000000+syn_nevents)
    syn_qlat = rng.uniform(syn_qlat_range[0], syn_qlat_range[1], syn_nevents)
    syn_qlon = rng.uniform(syn_qlon_range[0], syn_qlon_range[1], syn_nevents)
    syn_qdep = rng.uniform(syn_qdep_range[0], syn_qdep_range[1], syn_nevents)

    # generate qmag based on G-R law
    syn_qmag_possibilities = np.arange(syn_qmag_range[0], syn_qmag_range[1], 0.001)
    syn_qmag_a = 4.0
    syn_qmag_b = 0.4
    syn_qmag_possibilities = np.arange(syn_qmag_range[0], syn_qmag_range[1] + 0.002, 0.001)
    syn_qmag_probabilities = 10**(syn_qmag_a - syn_qmag_b * syn_qmag_possibilities)
    syn_qmag_probabilities = syn_qmag_probabilities / sum(syn_qmag_probabilities)
    syn_qmag = rng.choice(syn_qmag_possibilities, syn_nevents, p=syn_qmag_probabilities)

    syn_delsig_scatter = np.ones(syn_nevents)
    # syn_delsig_scatter = np.random.lognormal(0, 2, syn_nevents) # 0.473 is the std of real data
    syn_delsig_scatter = np.random.lognormal(0, 1.5, syn_nevents) # 0.473 is the std of real data

    syn_delsig = syn_delsig_fixed * syn_delsig_scatter

    # generate station-specific data using the above ranges
    syn_slat = rng.uniform(syn_slat_range[0], syn_slat_range[1], syn_nstations)
    syn_slon = rng.uniform(syn_slon_range[0], syn_slon_range[1], syn_nstations)
    syn_selev = rng.uniform(syn_selev_range[0], syn_selev_range[1], syn_nstations)
    syn_stid = np.arange(0, syn_nstations)
    syn_chnames = [str(el) for el in syn_stid]
    syn_station_names = np.array([".".join(["SN", f"{el.zfill(3)}", "", f"HH{components[0]}"]) for el in syn_chnames])
    syn_nts = np.array(np.round(rng.uniform(syn_nts_range[0], syn_nts_range[1], syn_nevents)), dtype=int)
    syn_nrecords = np.sum(syn_nts)

    print(f"{syn_nevents:,.0f} synthetic events")
    print(f"{syn_nstations:,.0f} synthetic stations")
    print(f"{syn_nrecords:,.0f} synthetic records")

else:
    print("No synthetic data will be generated.")

No synthetic data will be generated.


In [10]:

A =     np.array([4.33E5, 3.974E6])
Ap =    np.array([4.665E5, 3.938E6])

B =     np.array([4.40E5, 3.9343E6])
Bp =    np.array([4.60E5, 3.9543E6])

Define some functions

In [11]:

def get_colormap(d, cmap='coolwarm_r', method='percentile', **kwargs):

    d = d.astype(float)
    n = len(d)
    
    if method=='percentile':
        assert 'percentiles' in kwargs.keys(), "percentiles must be in kwargs"

        pct = kwargs['percentiles']

        d_sort = np.sort(d)

        cmap = plt.get_cmap(cmap)
        bounds = np.linspace(d_sort[int(pct[0]*n)], d_sort[int(pct[1]*n)], 7)
        norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

        smap = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    if method=='mediancenter':

        center = np.median(d)
        if 'percentiles' in kwargs.keys():
            pct = kwargs['percentiles']
        else:
            pct = [0.05, 0.95]
        
        d_sort = np.sort(d)

        center_dists = np.array(center - d_sort[int(pct[0]*n)], 
                                d_sort[int(pct[1]*n)] - center)
        dist = np.mean(center_dists)

        cmap = plt.get_cmap(cmap)
        bounds = np.linspace(center-dist, center+dist, 7)
        norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

        smap = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)

    if method=='logmediancenter':
        logd = np.log10(d)
        logcenter = np.median(logd)
        if 'percentiles' in kwargs.keys():
            pct = kwargs['percentiles']
        else:
            pct = [0.05, 0.95]
        
        logd_sort = np.sort(logd)

        center_dists = np.array(logcenter - logd_sort[int(pct[0]*n)], 
                                logd_sort[int(pct[1]*n)] - logcenter)
        dist = np.mean(center_dists)

        cmap = plt.get_cmap(cmap)
        bounds = np.logspace(logcenter-dist, logcenter+dist, 7)
        norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

        smap = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)

    return cmap, smap, norm



def p_norm(v, p):
    """Return the p-norm of a vector v (see ref. [2])

    Args:
        v (np.ndarray): array to calculate p-norm of
        p (int): p for p-norm

    Returns:
        float: p-norm of v
    """
    return np.power(np.sum(np.power(np.abs(v), p)), 1/p)

def model_p_norm(m, model_function, t, d, p):
    """Intermediate function for calculating the p-norm.

    Args:
        m (array-like): array of model parameters
        model_function (function): callable function
        t (np.ndarray): independent variable
        d (np.ndarray): data array
        p (int): p for p-norm

    Returns:
        float: p-norm of the residual (expected - observed) array
    """
    y = model_function(m, t)
    return p_norm(y - d, p)

def fit_p_norm(x, d, model, m0, p):
    """Find a fit to a model using scipy.optimize.minimize()

    Args:
        x (np.ndarray): independent variable
        d (np.ndarray): observations
        model (function): callable function, with inputs m (model 
            params) and x
        m0 (array-like): starting values for model parameters
        p (int): integer for choosing p-norm (1: L1-norm, 2: L2-norm, 
            etc.)

    Returns:
        np.ndarray: Best-fit model parameters
    """
    from scipy.optimize import minimize
    res = minimize(model_p_norm, m0, args=(model, x, d, p))
    return res.x

def line_model(m, x):
    """Slope-intercept form of a line.

    Args:
        m (array-like): (slope, intercept)
        x (np.ndarray): independent variable

    Returns:
        np.ndarray: y values on line
    """
    return m[0] * x + m[1]

def fit_line_p_norm(x, y, p):
    """Fit a simple line to x, y data using the p-norm.

    Args:
        x (np.ndarray): x values of data
        y (np.ndarray): y values of data
        p (np.ndarray): which norm to use to calculate the line 
            parameters. Least-squares when p==2.

    Returns:
        np.ndarray: (slope, intercept) of best-fit line.
    """
    m0 = np.polyfit(x, y, 1)
    return fit_p_norm(x, y, line_model, m0, p)

def compute_logbeta(spectra, low_f_ind, high_f_ind):
    # spectra is an (N x nf) array
    low_band = np.median(np.log10(spectra[:, low_f_ind[0]:low_f_ind[1]+1]), axis=1)
    high_band = np.median(np.log10(spectra[:, high_f_ind[0]:high_f_ind[1]+1]), axis=1)
    logbeta = high_band - low_band
    return logbeta

def boxplot(x, y, xbins, ax=None, color='r', **kwargs):

    if ax is None: ax = plt.gca()
    boxprops = dict(color=color)
    medianprops = dict(color=color, linewidth=2)
    whiskerprops = dict(color=color)
    capprops = dict(color=color)

    # toss out values of x, y where x is outside xbins bounds?
    
    inds = np.digitize(x, bins=xbins) - 1

    xmids = (xbins[1:] + xbins[:-1]) / 2

    xmids = xmids[np.unique(inds)]

    xwidth = xbins[1]-xbins[0]
    
    # get data in proper format for boxplot. This forms a sequence of 1D
    # arrays
    X = [y[inds==i] for i in np.unique(inds)]

    keys = kwargs.keys()
    if 'positions' not in keys: kwargs = {**kwargs, 'positions': xmids}
    if 'manage_ticks' not in keys: kwargs = {**kwargs, 'manage_ticks': False}
    if 'widths' not in keys: kwargs = {**kwargs, 'widths': xwidth*0.8}
    if 'sym' not in keys: kwargs = {**kwargs, 'sym': ''}
    if 'whis' not in keys: kwargs = {**kwargs, 'whis': (5, 95)}

    if 'boxprops' not in keys: kwargs = {**kwargs, 'boxprops': boxprops}
    if 'medianprops' not in keys: kwargs = {**kwargs, 'medianprops': medianprops}
    if 'whiskerprops' not in keys: kwargs = {**kwargs, 'whiskerprops': whiskerprops}
    if 'capprops' not in keys: kwargs = {**kwargs, 'capprops': capprops}

    out = ax.boxplot(X, **kwargs)

    return out

def compute_stn(signal_spectra, noise_spectra, stn_inds):
    # signal_spectra and noise_spectra are (N x nf) 2D-arrays
    signal = np.median(signal_spectra[:, stn_inds[0]:stn_inds[1]+1], axis=1)
    noise = np.median(noise_spectra[:, stn_inds[0]:stn_inds[1]+1], axis=1)
    stn = signal / noise
    return stn

def get_cha(stname):
    return stname.split('.')[-1]

def get_component(cha):
    return cha[-1]

def get_units(cha):
    return cha[-2]

def get_sdir(stname):
    if stname[-1] in ['1', 'Z']:
        return 'V'
    elif stname[-1] in ['2', '3', 'N', 'E']:
        return 'H'
    else:
        raise ValueError(f"Unknown component: {stname[-1]}")
    

def boxplot(x, y, xbins, ax=None, color='r', **kwargs):

    if ax is None: ax = plt.gca()
    boxprops = dict(color=color)
    medianprops = dict(color=color, linewidth=2)
    whiskerprops = dict(color=color)
    capprops = dict(color=color)

    # toss out values of x, y where x is outside xbins bounds?
    
    inds = np.digitize(x, bins=xbins) - 1

    xmids = (xbins[1:] + xbins[:-1]) / 2

    xmids = xmids[np.unique(inds)]

    xwidth = xbins[1]-xbins[0]
    
    # get data in proper format for boxplot. This forms a sequence of 1D
    # arrays
    X = [y[inds==i] for i in np.unique(inds)]

    keys = kwargs.keys()
    if 'positions' not in keys: kwargs = {**kwargs, 'positions': xmids}
    if 'manage_ticks' not in keys: kwargs = {**kwargs, 'manage_ticks': False}
    if 'widths' not in keys: kwargs = {**kwargs, 'widths': xwidth*0.8}
    if 'sym' not in keys: kwargs = {**kwargs, 'sym': ''}
    if 'whis' not in keys: kwargs = {**kwargs, 'whis': (5, 95)}

    if 'boxprops' not in keys: kwargs = {**kwargs, 'boxprops': boxprops}
    if 'medianprops' not in keys: kwargs = {**kwargs, 'medianprops': medianprops}
    if 'whiskerprops' not in keys: kwargs = {**kwargs, 'whiskerprops': whiskerprops}
    if 'capprops' not in keys: kwargs = {**kwargs, 'capprops': capprops}

    out = ax.boxplot(X, **kwargs)

    return out

def distance(s_lat, s_lng, e_lat, e_lng):

   # Approximate radius of earth in km
   R = 6373.0

   s_lat = s_lat*np.pi/180.0
   s_lng = np.deg2rad(s_lng)
   e_lat = np.deg2rad(e_lat)
   e_lng = np.deg2rad(e_lng)

   d = np.sin((e_lat - s_lat)/2)**2 + np.cos(s_lat)*np.cos(e_lat) * np.sin((e_lng - s_lng)/2)**2

   return 2 * R * np.arcsin(np.sqrt(d))


In [12]:
if generate_synthetic:
    fig, ax = plt.subplots(2,2, figsize=[8,8], layout='constrained')
    ax[0,0].hist(syn_qmag, bins=np.arange(syn_qmag_range[0]-1, syn_qmag_range[1]+1, 0.1), histtype='step', color='b')
    ax[0,0].set_yscale('log')
    ax[0,0].set_xlim(syn_qmag_range[0]-1, syn_qmag_range[1]+1)
    ax[0,0].set_xlabel("Catalog Magnitude")
    ax[0,0].set_ylabel("Count")

    ax[0,1].scatter(syn_qlon, syn_qlat, c='k', s=(syn_qmag**2)/20, edgecolors='none')
    ax[0,1].scatter(syn_slon, syn_slat, c='orange', s=50, marker='v', edgecolors='none')
    ax[0,1].set_xlabel("Longitude")
    ax[0,1].set_ylabel("Latitude")

    ax[1,0].scatter(syn_qmag, syn_delsig/1E6, c='k', s=1, edgecolors='none')
    ax[1,0].set_xlabel("Catalog Magnitude")
    ax[1,0].set_ylabel("Stress Drop (MPa)")
    ax[1,0].set_yscale('log')
    boxplot(x=syn_qmag, y=syn_delsig/1E6, xbins=np.arange(1, 10.1, 0.2), ax=ax[1,0], color='r')
    ax[1,0].set_yticks([0.1, 0.3, 1.0, 3.0, 10, 30, 100])
    ax[1,0].set_yticklabels([0.1, 0.3, 1.0, 3.0, 10, 30, 100])


    # histogram of delsigs
    ax[1,1].hist(syn_delsig/1E6, bins=np.logspace(-2, 5, 100), histtype='step', color='b')
    # ax[1,1].set_yscale('log')
    ax[1,1].set_xscale('log')
    ax[1,1].set_xlim(1E-2, 1E5)
    ax[1,1].set_xlabel("Stress Drop (MPa)")
    ax[1,1].set_ylabel("Count")

    plt.show()

Load .spec files into a DataFrame

In [13]:
# sp = 1 / (1 + (f / 4)**2)
# kappa = 0.02

# Af = 1 * np.exp(-np.pi * kappa * f)

# plt.figure()
# plt.plot(f, sp)
# plt.plot(f, Af)
# plt.plot(f, Af * sp)
# plt.xscale('log')
# plt.yscale('log')
# plt.show()

In [14]:
# np.reshape(sp, (len(f), 1))

In [15]:
# sp = 1 / (1 + (f / 4)**2)
# sp_acc = sp * (2*np.pi*f)**2
# kappa = 0.02
# A0 = 1

# Af = A0 * np.exp(-np.pi * kappa * f)
# Df = (A0 / (2*np.pi*f)**2) * np.exp(-np.pi * kappa * f)

# sp_logbeta = compute_logbeta(np.reshape(sp, (1, len(f))), low_beta_window_inds, high_beta_window_inds)[0]
# sp_acc_logbeta = compute_logbeta(np.reshape(sp_acc, (1, len(f))), low_beta_window_inds, high_beta_window_inds)[0]

# print(f"sp_logbeta = {sp_logbeta}")
# print(f"sp_acc_logbeta = {sp_acc_logbeta}")



# plt.figure()
# plt.plot(f, sp, label='sp (brune disp)')
# plt.plot(f, sp_acc, label='sp (brune acc)')
# plt.plot(f, Af, label='Af')
# plt.plot(f, Df, label='Df')
# plt.plot(f, sp*Df, label='sp*Df')
# plt.plot(f, sp_acc*Af, label='sp_acc*Af')
# plt.xlabel('Frequency (Hz)')
# plt.legend()
# plt.xscale('log')
# plt.yscale('log')
# plt.show()



In [16]:
# K = np.linspace(0.01, 0.3, 5)
# plt.figure()
# for i, kappa in enumerate(K):
#     Af = 1 * np.exp(-np.pi * kappa * f)
#     plt.plot(f, Af, label=f'kappa = {kappa:.2f}', c=(0,0,i/len(K)))
#     plt.plot(f, Af / sp, c=(i/len(K),0,0))
# # plt.plot(f, sp*(2*np.pi*f)**2)
# plt.xscale('log')
# plt.yscale('log')
# plt.xlabel('Frequency (Hz)')
# plt.legend()
# plt.show()

In [17]:
if not generate_synthetic:
    data_dir_files = os.listdir(spectra_dir)

    if phase=='p':
        pkl_name = 'p_spectra.pkl'
    elif phase=='s':
        pkl_name = 's_spectra.pkl'


    if 'df_all' in locals():
        print("df_all already in memory")
    else:
        if pkl_name in data_dir_files:
            # load pre-computed df_all DataFrame if possible
            print(f"Loading {pkl_name}...")
            t0 = time.time()
            df_all = pd.read_pickle(spectra_dir+pkl_name)
            print(f"    {len(df_all):,.0f} station-event pairs loaded from {pkl_name} (t = {time.time()-t0:.3f} s)")


        else:
            spec_files = [el for el in os.listdir(spec_dir) if '3' in el]
            nspec = len(spec_files)

            test = read_spec_max_df(spec_dir+spec_files[0])
            nf = len(test.loc[0, 's2'])
            f_load = np.linspace(0, f_nyquist, nf)
            f_load[0] = f_load[1] / 2


            # Read all spectra files in spec_dir
            print(f"Reading .spec files from {spec_dir}")
            print("----------------------------------------")
            d = [[]] * nspec
            for i in trange(nspec):
                filename = spec_files[i]
                d[i] = read_spec_max_df(spec_dir+filename)

                # if i >= 10: raise ValueError()
            print("Concatenating DataFrames...", end="")
            t0 = time.time()
            df_all = pd.concat(d)
            df_all.reset_index(drop=True, inplace=True)
            tf = time.time()
            print(f"Done ({tf-t0:.2f} s). {len(df_all):,.0f} records loaded. \n")
            ###

            # convert to displacement

            # vel_corr = (2 * np.pi * f_load)**-1
            # acc_corr = (2 * np.pi * f_load)**-2

            # for i, row in df_all.iterrows():
            #     unit = row['stid'].split('.')[-1][1]
            #     if unit=='N':
            #         corr = acc_corr
            #     elif unit=='H':
            #         corr = vel_corr
            #     df_all.at[i, 's1'] = row['s1'] * corr
            #     df_all.at[i, 's2'] = row['s2'] * corr


            # Save the information as a pickle object
            print(f"Saving combined DataFrame to {pkl_name}...", end="")
            t0 = time.time()
            df_all.to_pickle(spectra_dir+pkl_name)
            tf = time.time()
            print(f"Done ({tf-t0:.2f} s).")

            event_id = df_all['event_id'].values.astype(str)
            station_id = df_all['stid'].values
            nuniq = len(np.unique(event_id + station_id))
            assert nuniq == len(df_all), 'uh oh'

elif generate_synthetic:
    # generate a synthetic df_all DataFrame

    event_id = np.zeros(syn_nrecords, dtype=int)
    nts = np.zeros(syn_nrecords, dtype=int)
    qlat = np.zeros(syn_nrecords, dtype=float)
    qlon = np.zeros(syn_nrecords, dtype=float)
    qdep = np.zeros(syn_nrecords, dtype=float)
    qmag = np.zeros(syn_nrecords, dtype=float)
    stname = np.zeros(syn_nrecords, dtype='<U12')
    slat = np.zeros(syn_nrecords, dtype=float)
    slon = np.zeros(syn_nrecords, dtype=float)
    selev = np.zeros(syn_nrecords, dtype=float)
    deldist = np.zeros(syn_nrecords, dtype=float)
    delsig = np.zeros(syn_nrecords, dtype=float)
    fc = np.zeros(syn_nrecords, dtype=float)
    s1 = np.zeros((syn_nrecords, syn_nf), dtype=float)
    s2 = np.zeros((syn_nrecords, syn_nf), dtype=float)
    a2_max = np.zeros(syn_nrecords, dtype=float)

    spectrum_noise = np.ones((syn_nrecords, syn_nf), dtype=float)

    if add_spectrum_noise:
        spectrum_noise = np.abs(rng.normal(1, 0.5, (syn_nrecords, syn_nf)))

    n = 0
    for i in trange(syn_nevents):


        # for each event, choose syn_nts random unique stations
        stn_inds = rng.choice(syn_nstations, syn_nts[i], replace=False)

        # compute distance between event and station
        syn_deldist = distance(syn_qlat[i], syn_qlon[i], syn_slat[stn_inds], syn_slon[stn_inds])

        syn_M0 = np.power(10, (1.5 * syn_qmag[i] + 9.1))

        # add some scatter to delsig
        logomega0_syn = np.log10(syn_M0) - logomega0_corr

        # estimate fc given Mw_syn and assuming self-similarity
        syn_fc = np.power((16/7.0) * ((syn_delsig[i])/syn_M0), 1/3.0) * syn_k * syn_Vs
        brunes = np.tile((np.power(10, logomega0_syn) / (1 + np.power(syn_f[:, np.newaxis]/syn_fc, 2))), syn_nts[i]).T
        

        event_id[n:n+syn_nts[i]] = syn_event_id[i]
        nts[n:n+syn_nts[i]] = syn_nts[i]
        qlat[n:n+syn_nts[i]] = syn_qlat[i]
        qlon[n:n+syn_nts[i]] = syn_qlon[i]
        qdep[n:n+syn_nts[i]] = syn_qdep[i]
        qmag[n:n+syn_nts[i]] = syn_qmag[i]

        stname[n:n+syn_nts[i]] = syn_station_names[stn_inds]
        slat[n:n+syn_nts[i]] = syn_slat[stn_inds]
        slon[n:n+syn_nts[i]] = syn_slon[stn_inds]
        selev[n:n+syn_nts[i]] = syn_selev[stn_inds]
        deldist[n:n+syn_nts[i]] = syn_deldist
        delsig[n:n+syn_nts[i]] = syn_delsig[i]

        fc[n:n+syn_nts[i]] = syn_fc
        a2_max[n:n+syn_nts[i]] = np.max(brunes, axis=1)

        # these should be unique in real data for each record, but for now they are the same for each event
        # in the future, we could add an attenuation effect
        s2[n:n+syn_nts[i], :] = brunes * spectrum_noise[n:n+syn_nts[i], :]
        s1[n:n+syn_nts[i], :] = brunes / 1E5

        n += syn_nts[i]

    df_all = pd.DataFrame({
        "event_id": event_id,
        "nts": nts,
        "qlat": qlat,
        "qlon": qlon,
        "qdep": qdep,
        "qmag": qmag,
        "stname": stname,
        "slat": slat,
        "slon": slon,
        "selev": selev,
        "deldist": deldist,
        "delsig": delsig,
        "fc": fc,
        "s1": list(s1),
        "s2": list(s2),
        "a2_max": a2_max
    })
    ds = df_all[['event_id', 'delsig', 'fc', 'qlat', 'qlon', 'qmag']].groupby('event_id').first().reset_index()

    # write ds to fwf file
    np.savetxt(f'/Users/ivandevert/projects/spectral_falloff_ratio/data/beta/ds_{syn_dataset_name}.txt', ds.values, fmt='%8i %15.9e %8.4f %15.11f %15.11f %5.2f')


Reading .spec files from /Users/ivandevert/projects/ridgecrest2019/proc/specdecomp_s/01_compspec/SPECOUT/
----------------------------------------


100%|██████████| 12936/12936 [02:51<00:00, 75.46it/s]


Concatenating DataFrames...Done (3.85 s). 2,856,948 records loaded. 

Saving combined DataFrame to s_spectra.pkl...Done (68.17 s).


In [ ]:
# nf = len(df_all.loc[0, 's2'])

# f = np.linspace(0, f_nyquist, nf)

# vel_corr = (2 * np.pi * f_load)**-1
# acc_corr = (2 * np.pi * f_load)**-2

# for i, row in df_all.iterrows():
#     unit = row['stid'].split('.')[-1][1]
#     if unit=='N':
#         row['s2'] = row['s2'] * vel_corr
#     elif unit=='H':
#         row['s2'] = row['s2'] * acc_corr

#     # print(unit)

In [ ]:
# plot a few s2 examples
if generate_synthetic:
    i = 0
    j = 0
    plt.figure()
    plt.plot(syn_f, s2[:100, :].T)
    plt.yscale('log')
    plt.xscale('log')
    plt.show()

In [ ]:
# Print out frequency information

nf = len(df_all.loc[0, 's2'])

f = np.linspace(0, f_nyquist, nf)
dfreq = f[1] - f[0]


# calculate actual indices and bands for beta computation
low_beta_window_inds = np.array([np.argmin(np.abs(f - el)) for el in low_beta_window])
high_beta_window_inds = np.array([np.argmin(np.abs(f - el)) for el in high_beta_window])

low_beta_band = [f[low_beta_window_inds[0]], f[low_beta_window_inds[1]]]
high_beta_band = [f[high_beta_window_inds[0]], f[high_beta_window_inds[1]]]


# calculate actual indices and bands for STN computations
stn_inds = [np.argmin(np.abs(f - el)) for el in stn_f_range]

stn_band = [f[stn_inds[0]], f[stn_inds[1]]]


print("")
print("FREQUENCY ARRAY INFORMATION")
print("----------------------------")
print(f"Frequency array ranges from {f[0]:.2f} to {f[-1]:.2f} Hz with {len(f)} elements (df = {dfreq:.3f} Hz). ")
print(f"Desired | Actual low-frequency band:   {low_beta_window[0]:7.3f} -{low_beta_window[1]:7.3f} Hz | {low_beta_band[0]:7.3f} -{low_beta_band[1]:7.3f} Hz")
print(f"Desired | Actual high-frequency band:  {high_beta_window[0]:7.3f} -{high_beta_window[1]:7.3f} Hz | {high_beta_band[0]:7.3f} -{high_beta_band[1]:7.3f} Hz")
print(f"Desired | Actual signal-to-noise band: {stn_f_range[0]:7.3f} -{stn_f_range[1]:7.3f} Hz | {stn_band[0]:7.3f} -{stn_band[1]:7.3f} Hz")


Load data, index events and stations, and filter out bad results. This is to setup for the inverse problem.

In [ ]:


# These are columns that are station-dependent, event-dependent, and 
# both-dependent. pandas pivot_table might be useful here
st_dep = ['stname', 'slat', 'slon', 'selev']
ev_dep = ['event_id', 'qmag', "qlon", "qlat", "qdep"]
dependents = ['deldist', 's1', 's2', 'logbeta']

print("-----------------------------")
print("---    DATA PROCESSING    ---")
print("-----------------------------")
    
print("\nFiltering out records based on user-specified criteria...")
# print(f" --- Using {phase.upper()}-waves, so {','.join(components)} components")
# print(f" --- Using {','.join(units)} units")
# print(f" --- Maximum event-station horizontal distance: {dist_max} km")
# print(f" --- Calibration events within {calib_rmax} km horizontally and {calib_zmax} km vertically")

print("---------------------------------------------------------")

# df/df_all has one row per station-event pair
df = df_all.copy()
df = df.rename(columns={"stid":"stname"})

# df = df.drop(columns=['nts_p', 'nts_s'])

# get initial number of records
npairs_initial = len(df)
nevents_initial = len(np.unique(df['event_id']))
nstations_initial = len(np.unique(df['stname']))
print(f"{npairs_initial:,.0f} records in total. {nevents_initial:,.0f} events, {nstations_initial:,.0f} stations.")

# # remove events below the lower calibration event magnitude, since these are unused for all future steps
# t0 = time.time()
# df = df[df['qmag'] >= calib_mag_range[0]].reset_index(drop=True)
npairs = len(df)
# print(f"{npairs_initial - npairs:,.0f} pairs with qmag < M{calib_mag_range[0]:.2} removed (t = {time.time()-t0:.3f} s). {len(df):,.0f} remaining.")

# remove distant event-station records
t0 = time.time()
df = df[df['deldist'] <= dist_max].reset_index(drop=True)
print(f"{npairs - len(df):,.0f} pairs with deldist > {dist_max} removed (t = {time.time()-t0:.3f} s). {len(df):,.0f} remaining.")

# get cha, component, and unit columns
t0 = time.time()
df['cha'] = df['stname'].apply(get_cha)
df['component'] = df['cha'].apply(get_component)
df['unit'] = df['cha'].apply(get_units)
print(f"Computed cha, component, and unit columns (t = {time.time()-t0:.2} s). {len(df):,.0f} remaining.")

# remove components not in 'components' object ('N', 'E', 'Z', for example)
t0 = time.time()
l1 = len(df)
nstations = len(np.unique(df['stname']))
df = df[np.any([df['component'].values==el for el in components], axis=0)].reset_index(drop=True)
print(f"{l1-len(df):,.0f} records, {nstations-len(np.unique(df['stname']))} stations removed (component not in {components}) (t = {time.time()-t0:.2} s). {len(df):,.0f} remaining.")

# add a column 'sdir' for direction, either 'H' or 'V', for horizontal or 
# vertical, based on the last character of the stname. 'N', 'E', '2', and '3'
# should be horizontal, '1' and 'Z' should be vertical


# t0 = time.time()
# df['sdir'] = df['stname'].apply(get_sdir)
# print(f"Computed sdir column (t = {time.time()-t0:.2} s). {len(df[df['sdir']=='H']):,.0f} horizontal, {len(df[df['sdir']=='V']):,.0f} vertical records")
# st_dep += ['sdir']

nevents = len(np.unique(df['event_id']))
nstations = len(np.unique(df['stname']))
print(f"{nevents:,.0f} events remaining, {nstations:,.0f} stations remaining.")

# remove units not in 'unit' object ('H', 'N', for example)
t0 = time.time()
l1 = len(df)
df = df[np.any([df['unit'].values==el for el in units], axis=0)].reset_index(drop=True)
print(f"{l1-len(df):,.0f} records removed (unit not in {units}) (t = {time.time()-t0:.2} s). {len(df):,.0f} remaining.")



# drop these columns
df = df.drop(columns=['cha', 'component', 'unit']).reset_index(drop=True)

# # # explode!
df['event_id'] = df['event_id'].astype(int)
df['qmag'] = df['qmag'].astype(float)
df['qlat'] = df['qlat'].astype(float)
df['qlon'] = df['qlon'].astype(float)
df['qdep'] = df['qdep'].astype(float)
df['slat'] = df['slat'].astype(float)
df['slon'] = df['slon'].astype(float)
df['selev'] = df['selev'].astype(float)
df['deldist'] = df['deldist'].astype(float)


df['logbeta'] = np.nan

# Compute sx and sy (station easting and northing)
t0 = time.time()
df['sx'], df['sy'], zn, zl = utm.from_latlon(
    df['slat'].values, 
    df['slon'].values
    )
st_dep += ['sx', 'sy']
df.reset_index(drop=True, inplace=True)

# Compute qxs and qys of events (event eastings and northings)
df['qx'], df['qy'], zn, zl = utm.from_latlon(
    df['qlat'].values, 
    df['qlon'].values
    )
ev_dep += ['qx', 'qy']
df.reset_index(drop=True, inplace=True)
print(f"Earthquake and station eastings and westings computed (t = {time.time()-t0:.3f} s)")

# # make sure each earthquake has enough remaining records
# df_ev = df.groupby(ev_dep, as_index=False)[st_dep+dependents].agg(list)
# df_ev = df_ev[df_ev['stname'].map(len) >= nrecords_min].reset_index(drop=True)
# print(f"{int(nevents_initial-len(df_ev))} events removed due to not enough"
#     f" records (needs >= {nrecords_min})")

# group by station in df_sta and add 'stind' column, unique for each station
t0 = time.time()
df_sta = df.groupby(st_dep, as_index=False)[ev_dep+dependents].agg(list)
df_sta['stind'] = df_sta.index.values.astype(int)
st_dep += ['stind']
print(f"Grouped by station (t = {time.time()-t0:.3f} s). {len(df_sta):,.0f} stations remaining.")

t0 = time.time()
df = df_sta.explode(ev_dep + dependents)
df.reset_index(drop=True, inplace=True)
print(f"Exploded (t = {time.time()-t0:.3f} s)")

df['event_id'] = df['event_id'].astype(int)
df['qmag'] = df['qmag'].astype(float)
df['qlat'] = df['qlat'].astype(float)
df['qlon'] = df['qlon'].astype(float)
df['qdep'] = df['qdep'].astype(float)
df['slat'] = df['slat'].astype(float)
df['slon'] = df['slon'].astype(float)
df['selev'] = df['selev'].astype(float)
df['deldist'] = df['deldist'].astype(float)
df['qx'] = df['qx'].astype(float)
df['qy'] = df['qy'].astype(float)
df['sx'] = df['sx'].astype(float)
df['sy'] = df['sy'].astype(float)
df['logbeta'] = df['logbeta'].astype(float)




## Compute logbeta values for spectra

spec_inds = np.where([hasattr(el, '__len__') for el in df['s2'].values ])[0]

# compute logbeta for spectra
print("Computing logbeta for spectra...", end='')
t0 = time.time()
s2 = np.vstack(df['s2'].values[spec_inds], dtype=float)

logbeta = compute_logbeta(s2, low_beta_window_inds, high_beta_window_inds)
df['logbeta'] = np.nan
df.loc[spec_inds, 'logbeta'] = logbeta
print(f"done (t = {time.time()-t0:.3f} s)")

if 'logbeta' not in dependents: dependents += ['logbeta']



## Compute STN values for spectra
# find indices where spectra are non-NaN, stack into 2D-array
# should get equivalent result using either p1 or p2 columns
print("Computing STN for spectra...", end='')
t0 = time.time()
spec_inds = np.where([hasattr(el, '__len__') for el in df['s2'].values ])[0]
s2 = np.vstack(df['s2'].values[spec_inds], dtype=float)
s1 = np.vstack(df['s1'].values[spec_inds], dtype=float)

# compute stn for each row in the 2D-array
stn = compute_stn(s2, s1, stn_inds)
df['stn'] = np.nan
df.loc[spec_inds, 'stn'] = stn
print(f"done (t = {time.time()-t0:.3f} s)")

if 'stn' not in dependents: dependents += ['stn']

df = df[df['stn'] > stn_req].reset_index(drop=True)


# group by event
df_ev = df.groupby(ev_dep, as_index=False)[st_dep+dependents].agg(list)

df_sta = df.groupby(st_dep, as_index=False)[ev_dep+dependents].agg(list)

nevents = len(np.unique(df['event_id']))
nstations = len(np.unique(df['stname']))

# get a list of event ids that are in df['event_id'] but not in df_ev['event_id'] 
absent = df[~df['event_id'].isin(df_ev['event_id'])].reset_index(drop=True)

assert nevents-len(df_ev)==0, 'Uh oh: missing events'
assert len(np.unique(absent['event_id']))==0, 'Uh oh: missing events'

# nevents = len(np.unique(df['event_id']))
# evs1 = df.copy()
# nstations = len(np.unique(df['stname']))
# print(f"{nevents:,.0f} events remaining, {nstations:,.0f} stations remaining.")


# save the events within calibration M range for later
df_calib = df[np.logical_and(df['qmag'] >= calib_mag_range[0], df['qmag'] < calib_mag_range[1])].reset_index(drop=True)

nevents = len(df_ev)
nstations = len(df_sta)
ndata = len(df)

print("-------------")
print("Final counts:")
print("-------------")
print(f"{nevents:,.0f} of {nevents_initial:,.0f} events ({nevents/nevents_initial*100:.1f}%)")
print(f"{nstations:,.0f} of {nstations_initial:,.0f} stations ({nstations/nstations_initial*100:.1f}%)")
print(f"{ndata:,.0f} of {npairs_initial:,.0f} records ({ndata/npairs_initial*100:.1f}%)")
print("")
print("Calibration events:")
print("-------------------")
print(f"{len(df_calib):,.0f} records of {len(np.unique(df_calib['event_id'])):,.0f} events between M {calib_mag_range[0]:.2f} and {calib_mag_range[1]:.2f}")


In [ ]:
# cha = [el.split('.')[-1] for el in df_calib['stname'].values]
# c = [[]]*len(cha)
# for i in range(len(cha)):
#     if cha[i]=='HHZ': c[i] = 'k'
#     elif cha[i]=='HNZ': c[i] = 'b'
#     elif cha[i]=='EHZ': c[i] = 'r'
#     else: print(cha[i])

# plt.figure()
# plt.scatter(df_calib['deldist'].values, df_calib['logbeta'].values, c=c, s=1, marker='.')
# plt.xlabel('Distance (km)')
# plt.ylabel('log(beta)')
# # plt.xscale('log')
# plt.plot(x, MM*x+y0, c='b')
# plt.show()

In [ ]:
minrange = 30
mindist = 10

df_calib_test = df_calib.copy()

df_calib_test = df_calib_test[df_calib_test['deldist'] > mindist].reset_index(drop=True)

# plot logbeta vs deldist for three example stations
df_sta_calib = df_calib_test.groupby(st_dep, as_index=False)[ev_dep+dependents].agg(list)

df_sta_calib

M = np.zeros((len(df_sta_calib), 2), dtype=float)
N = np.zeros(len(df_sta_calib), dtype=float)
D = np.zeros(len(df_sta_calib), dtype=float)
R = np.zeros(len(df_sta_calib), dtype=float)

# store the slopes and intercepts for each station's calibration events
for i in range(len(df_sta_calib)):
    row = df_sta_calib.iloc[i]

    deldist = np.array(row['deldist'])
    logbeta = np.array(row['logbeta'])
    N[i] = len(deldist)
    D[i] = np.median(deldist)
    R[i] = np.max(deldist) - np.min(deldist)
    m = fit_line_p_norm(deldist, logbeta, 1)
    M[i,0] = m[0]
    M[i,1] = m[1]

#     if i<10: 
#         plt.figure()
#         plt.scatter(row['deldist'], row['logbeta'], c='k', s=1, marker='.')
#         plt.plot(deldist, m[0]*deldist+m[1], c='r')
#         # plt.plot(deldist, m[0]*deldist+m[1], c='r')
#         plt.xlabel('Distance (km)')
#         plt.ylabel('logbeta')
#         plt.title(row['stname'])
#         plt.xlim([0, 110])
#         plt.ylim([-2, 1.5])
# plt.show()

isgood = np.logical_and(N > 200, R > minrange)

# scatter slope vs distance for good data
plt.figure()
plt.scatter(D, M[:,0], c='k', s=1, marker='.')
plt.scatter(D[isgood], M[isgood,0], c='r', s=20, marker='.')

# plot axhline at median slope of good and all data
plt.axhline(np.median(M[:,0]), c='k', ls='--')
plt.axhline(np.median(M[isgood,0]), c='r')
plt.xlabel('Distance (km)')
plt.ylabel('Slope')
# plt.xscale('log')
plt.ylim([-0.05, 0.05])
plt.show()

plt.figure()
plt.scatter(D, M[:,1], c='k', s=1, marker='.')
plt.scatter(D[isgood], M[isgood,1], c='r', s=20, marker='.')
plt.axhline(np.median(M[:,1]), c='k', ls='--')
plt.axhline(np.median(M[isgood,1]), c='r')
plt.xlabel('Distance (km)')
plt.ylabel('y-intercept')
# plt.xscale('log')
plt.ylim([-1.5, 1.5])
plt.show()

# plot slope vs intercept of good and all data
plt.figure()
plt.scatter(M[:,0], M[:,1], c='k', s=1, marker='.')
plt.scatter(M[isgood,0], M[isgood,1], c='r', s=20, marker='.')
plt.xlabel('Slope')
plt.ylabel('y-intercept')
# plt.xscale('log')
plt.xlim([-0.05, 0.05])
plt.ylim([-1.5, 1.5])
plt.show()



cha = [el.split('.')[-1] for el in df_sta_calib['stname'].values]
c = [[]]*len(cha)
for i in range(len(cha)):
    if cha[i]=='HHZ': c[i] = 'k'
    elif cha[i]=='HNZ': c[i] = 'b'
    elif cha[i]=='EHZ': c[i] = 'r'
    else: print(cha[i])

plt.figure()
plt.scatter(D, M[:,0], c=c, s=20, marker='o')
plt.scatter(D[isgood], M[isgood,0], c='lime', s=10, marker='.')
plt.xlabel('Distance (km)')
plt.ylabel('slope')
plt.ylim([-0.05, 0.05])
plt.show()

print(np.mean(M[isgood,0]))

In [ ]:
row

In [ ]:
# plot locations of row['sx'], row['sy'] and row['qx'], row['qy']

plt.figure()
plt.scatter(np.array(row['sx'])/1000, np.array(row['sy'])/1000, c='r', s=20, marker='v')
plt.scatter(np.array(row['qx'])/1000, np.array(row['qy'])/1000, c='k', s=20, marker='.')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()

In [ ]:
# new method of finding slopes

# first, setup grid of possible slopes
nM = 101
M = np.linspace(-0.3, 0.3, nM)

# for explanation plots
x = np.linspace(0, 110, 100)


# For each slope: 
# 1) compute residuals of data points with line of slope M[i] and y-intercept 0
# 2) compute mean of residuals (maybe median?)
# 3) store y-intercepts
# 4) compute sum of square

Y0 = np.zeros((nM, len(df_sta_calib)), dtype=float)
errs = np.zeros(nM)
for i in trange(nM):
    for j, row in df_sta_calib.iterrows():
        

        # Store x and y data arrays
        deldist = np.array(row['deldist'], dtype=float)
        logbeta = np.array(row['logbeta'], dtype=float)
        
        # Compute residuals
        res = M[i]*deldist - logbeta

        # Store y-intercept
        Y0[i, j] = np.mean(res)

        # Compute shifted residuals (res2 should have zero mean)
        res2 = res - Y0[i, j]

        # Store the sum of error**2
        errs[i] += np.sum(res2**2)

imin = np.argmin(errs)

# fit a parabola to nearest +/- 20 points around minimum using np.polyfit
X = M[imin-20:imin+20]
Y = errs[imin-20:imin+20]
p = np.polyfit(X, Y, 2)
mfit = -p[1]/(2*p[0])

print(f"Best-fit slope: {mfit:7.5e}")

# recompute y0 using this best-fit slope
y0 = np.zeros(len(df_sta_calib), dtype=float)
nev = np.zeros(len(df_sta_calib), dtype=int)
err = 0
for j, row in df_sta_calib.iterrows():
    # Store x and y data arrays
    deldist = np.array(row['deldist'], dtype=float)
    logbeta = np.array(row['logbeta'], dtype=float)
    nev[j] = len(deldist)
    
    # Compute residuals
    res = logbeta - mfit*deldist

    # Store y-intercept
    y0[j] = np.mean(res)

    # Compute shifted residuals (res2 should have zero mean)
    res2 = res - y0[j]

    # Store the sum of error**2
    err += np.sum(res2**2)

    if j < 10:
        plt.figure()
        plt.plot(f, np.array(row['s2']).T, c='grey', lw=1, label='Calibration events')
        plt.plot(f, np.mean(np.array(row['s2']), axis=0), c='k', lw=2, label='Mean spectrum')
        plt.xscale('log')
        plt.yscale('log')

        # title: station_name | n= 
        plt.title(f"{row['stname']} | n = {len(deldist)}")
        plt.xlabel('Frequency (Hz)')
        handles, labels = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        plt.legend(by_label.values(), by_label.keys())
        plt.show()

    # if y0[j] > 1:
    #     plt.figure()
    #     plt.scatter(deldist, logbeta, c='k', s=1, marker='.')
    #     plt.plot(x, mfit*x + y0[j], c='b', lw=2)
    #     plt.axhline(y0[j], c='r', ls='--')
    #     plt.xlabel('Distance (km)')
    #     plt.ylabel('log(beta)')
    #     plt.xlim([0, 110])
    #     plt.title(f"nev = {nev[j]}")
    #     plt.show()


In [ ]:
# compute estimate of Q
f_h = np.mean(high_beta_window)
f_l = np.mean(low_beta_window)

c = 6.000
Q = (f_h-f_l) * (-np.pi * np.log10(np.exp(1)) / (c * mfit))
print(Q)


In [ ]:

# plt.figure()
# plt.plot(M, errs, c='k')
# plt.axvline(mbest, c='r')
# plt.scatter(mbest, errs[imin], c='r', label='Best-fit slope: {:.4e}'.format(mbest))
# plt.plot(X, p[0]*X**2 + p[1]*X + p[2], c='b', lw=2, label='Parabola fit: {:.4e}'.format(mbest_fit))
# plt.xlabel('Slope')
# plt.ylabel('sum of error**2')
# plt.legend()
# plt.show()


In [ ]:
low_beta_window_inds

In [ ]:
low_beta_window_inds

In [ ]:
# peter's code
# convert beta value to kappa value
import numpy as np

def brune(fc, f):
   amp = 1./(1 + (f/fc)**2)
   return amp 

f1 = 3.0         #low-frequency reference
f2 = 18.5        #high-frequency reference

fc = 30.         #Brune model corner frequency

brunedif = np.log10(brune(fc, f2)) - np.log10(brune(fc, f1))
brunedif2 = np.log10(np.mean(brune(fc, f)[high_beta_window_inds[0]:high_beta_window_inds[1]+1])) - np.log10(np.mean(brune(fc, f)[low_beta_window_inds[0]:low_beta_window_inds[1]+1]))
print ("brunedif = ", brunedif)
print ("brunedif2 = ", brunedif2)

logbeta = 0.11         #possible value of logbeta

logbeta_cor = logbeta - brunedif         #Brune-model-corrected logbeta
print ("logbeta_cor = ", logbeta_cor)

factor = -np.pi*np.log10(np.e)
print ("factor = ", factor)

kappa = logbeta_cor/(factor*(f2 - f1))
print ("kappa = ", kappa)

In [ ]:
(-np.pi * np.log10(np.e))

In [ ]:
fc_fixed = 30
omega0 = 1

f_h = np.mean(high_beta_window)
f_l = np.mean(low_beta_window)

# f_h = 10**np.mean(np.log10(np.array(high_beta_window)))
# f_l = 10**np.mean(np.log10(np.array(low_beta_window)))

print(f_h, f_l, f_h-f_l)

spectra = (omega0) / (1 + (f/fc_fixed)**2)
spectra = spectra.reshape((1, len(spectra)))

s_half =  (omega0) / (1 + (f/(fc_fixed/2))**2)
s_half = s_half.reshape((1, len(s_half)))

s_double = (omega0) / (1 + (f/(fc_fixed*2))**2)
s_double = s_double.reshape((1, len(s_double)))



source_logbeta = compute_logbeta(spectra, low_beta_window_inds, high_beta_window_inds)[0]
source_logbeta_half = compute_logbeta(s_half, low_beta_window_inds, high_beta_window_inds)[0]
source_logbeta_double = compute_logbeta(s_double, low_beta_window_inds, high_beta_window_inds)[0]
print(source_logbeta, source_logbeta_half, source_logbeta_double)

# print(y0 - source_logbeta)

logbeta_0 = y0

kappa0 =   (logbeta_0 - source_logbeta) / (-np.pi * np.log10(np.e) * (f_h - f_l))
kappa0_1 = (logbeta_0 - source_logbeta_half) / (-np.pi * np.log10(np.e) * (f_h - f_l))
kappa0_2 = (logbeta_0 - source_logbeta_double) / (-np.pi * np.log10(np.e) * (f_h - f_l))




for j, row in df_sta_calib.iterrows():
    print(f"nev: {nev[j]:4d}, kappa0: {kappa0[j]:.3f}", row['stname'])

In [ ]:
bins = np.linspace(-3, 3, 51)

plt.figure()
plt.hist(y0, bins=bins, histtype='step', color='k')
plt.hist(y0[nev>10], bins=bins, histtype='step', color='r', label='nev > 10')
plt.axvline(0, c='k', lw=2)
plt.xlabel('y0')
plt.ylabel('Count')
plt.legend()
plt.show()

bins = np.linspace(-0.05, 0.15, 51)

plt.figure()
plt.hist(kappa0, bins=bins, histtype='step', color='k')
plt.hist(kappa0[nev>10], bins=bins, histtype='step', color='r', label='nev > 10')
# plt.hist(kappa0_double, bins=50, histtype='step', color='r')
# plt.hist(kappa0_half, bins=50, histtype='step', color='b')

plt.axvline(0, c='k', lw=2)
plt.legend()
plt.xlabel('kappa0')
plt.ylabel('Count')
plt.show()

In [ ]:
mbest = -p[1]/(2*p[0])
mbest

In [ ]:
plt.figure()
plt.hist(M[:,1], histtype='step', bins=np.arange(-10, 10, 0.1), color='k')
plt.hist(Y0, histtype='step', bins=np.arange(-1.5, 1.5, 0.1), color='r')
plt.xlim([-3, 3])
plt.show()


In [ ]:
df_sta_calib

In [ ]:
# histogram of R
plt.figure()
plt.hist(R, bins=20)
plt.xlabel('Range (km)')
plt.ylabel('Count')
plt.show()

In [ ]:
MM = np.mean(M[isgood,0])
x = np.arange(0, 110, 10)
Y0 = np.zeros(len(df_sta_calib), dtype=float)

for i in range(len(df_sta_calib)):
    row = df_sta_calib.iloc[i]

    deldist = np.array(row['deldist'])
    logbeta = np.array(row['logbeta'])

    md = np.median(deldist)
    ml = np.median(logbeta)
    y0 = ml - MM*md
    Y0[i] = y0
    # fit a line to deldist, logbeta with a given slope of MM
    

    m = fit_line_p_norm(deldist, logbeta, 1)

    # if N[i] > 200: 
    # plt.figure()
    # plt.scatter(row['deldist'], row['logbeta'], c='k', s=1, marker='.')
    # plt.plot(deldist, m[0]*deldist+m[1], c='r')
    # plt.plot(x, MM*x+y0, c='b')
    # plt.xlabel('Distance (km)')
    # plt.ylabel('logbeta')
    # plt.title(row['stname'])
    # plt.xlim([0, 110])
    # plt.ylim([-2, 1.5])
    # plt.show()
    # raise ValueError()
# plt.show()

# plot a histogram of Y0
plt.figure()
plt.hist(Y0, bins=30)
plt.xlabel('y-intercept')
plt.ylabel('Count')
plt.show()


In [ ]:
# Make a map view plot of stations colored by the Y0 values
# df_sta_calib['y0'] = Y0


elat_range = (df_all['qlat'].min(), df_all['qlat'].max())
elon_range = (df_all['qlon'].min(), df_all['qlon'].max())




plt.figure()
plt.scatter(df_sta_calib['slon'].values, df_sta_calib['slat'].values, c=kappa0, s=60, marker='v', cmap='seismic', edgecolors='k', vmin=0, vmax=0.1)
cbar = plt.colorbar()
cbar.set_label('kappa0')

baddata = kappa0<0
plt.scatter(df_sta_calib['slon'].values[baddata], df_sta_calib['slat'].values[baddata], c='r', s=60, marker='x', edgecolors='none', label=r"$\kappa_0 < 0$")
plt.plot([elon_range[0], elon_range[0], elon_range[1], elon_range[1], elon_range[0]], [elat_range[0], elat_range[1], elat_range[1], elat_range[0], elat_range[0]], 'r-', label='Extent of calib. evs')

plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.legend()
plt.show()

In [ ]:
# get the range of event lat and lons from df_all and store in two tuples (elat_range, elon_range)


plt.figure()
plt.scatter(df_sta_calib['slon'].values, df_sta_calib['slat'].values, c=nev, s=60, marker='v', cmap='YlGn', edgecolors='k', vmin=0, vmax=50)
cbar = plt.colorbar()
# label colorbar 'nevents'

cbar.set_label('Number of Events')


# draw a red box around elat_range and elon_range
plt.plot([elon_range[0], elon_range[0], elon_range[1], elon_range[1], elon_range[0]], [elat_range[0], elat_range[1], elat_range[1], elat_range[0], elat_range[0]], 'r-', label='Extent of calib. evs')

baddata = nev < 10
plt.scatter(df_sta_calib['slon'].values[baddata], df_sta_calib['slat'].values[baddata], c='orange', s=60, marker='v', label=r'$nev < 10$')


baddata = kappa0<0
plt.scatter(df_sta_calib['slon'].values[baddata], df_sta_calib['slat'].values[baddata], c='r', s=60, marker='x', label=r'$\kappa_0 < 0$')



plt.legend(loc='upper left')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.show()

In [ ]:
plt.figure()
plt.scatter(df_sta_calib['slon'].values[nev>=10], df_sta_calib['slat'].values[nev>=10], c=kappa0[nev>=10], s=60, marker='v', cmap='seismic', edgecolors='k', vmin=0, vmax=0.1)
cbar = plt.colorbar()
cbar.set_label('kappa0')
plt.plot([elon_range[0], elon_range[0], elon_range[1], elon_range[1], elon_range[0]], [elat_range[0], elat_range[1], elat_range[1], elat_range[0], elat_range[0]], 'r-', label='Extent of calib. evs')

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('nev >= 10')
plt.legend()
plt.show()

In [ ]:
# compare with Ahn data

ahn_stname = ["CI.CCA" ,
"CI.CCC",
"CI.CGO",
"CI.CLC",
"CI.CWC",
"CI.DAW",
"CI.DTP",
"CI.GSC",
"CI.ISA",
"CI.JRC2" ,
"CI.LRL",
"CI.MPM",
"CI.SLA",
"CI.SRT",
"CI.TEH",
"CI.TOW2" ,
"CI.WAS2" ,
"CI.WBM",
"CI.WBP",
"CI.WBS",
"CI.WCS2" ,
"CI.WHF",
"CI.WMF",
"CI.WOR",
"CI.WRC2" ,
"CI.WRV2" ,
"CI.WVP2" ,
"GS.CA01" ,
"GS.CA02" ,
"GS.CA03" ,
"GS.CA04" ,
"GS.CA05" ,
"GS.CA06" ,
"GS.CA07" ,
"GS.CA08" ,
"GS.CA09" ,
"NN.QSM",
"NP.5419" ,
"ZY.SV01" ,
"ZY.SV02" ,
"ZY.SV03" ,
"ZY.SV04" ,
"ZY.SV05" ,
"ZY.SV06" ,
"ZY.SV07" ,
"ZY.SV08"]
ahn_kappa0 = "0.0582 0.055 0.0476 0.0239 0.0287 0.0232 0.0283 0.0324 0.0207 0.0295 0.0281 0.0285 0.0469 0.0432 0.0386 0.0488 0.0412 0.0365 0.0242 0.0291 0.0539 0.038 0.0342 0.0238 0.0515 0.0299 0.0361 0.0382 0.0564 0.0436 0.0454 0.0658 0.0375 0.0405 0.0294 0.0457 0.0416 0.0416 0.0546 0.0476 0.0467 0.0523 0.0619 0.0301 0.0316 0.0417"
# split by whitespace make numpy float array
ahn_kappa0 = np.array(ahn_kappa0.split(), dtype=float)

# make dataframe
df_ahn = pd.DataFrame({'stname': ahn_stname, 'kappa0_ahn': ahn_kappa0})

# make dataframe from kappa0
stname = ['.'.join(el.split('.')[:-2]) for el in df_sta_calib['stname'].values]
df_mydata = pd.DataFrame({'stname': stname, 'kappa0': kappa0, 'kappa0_1': kappa0_1, 'kappa0_2': kappa0_2})

# merge on stname column
df_both = pd.merge(df_ahn, df_mydata, how='inner', on='stname')
df_both

# scatter plot compare kappa0 values
plt.figure()
plt.scatter(df_both['kappa0_ahn'].values, df_both['kappa0'].values, c='k', s=5, marker='.', edgecolors='k')

# plot x=y line
plt.plot([0, 0.15], [0, 0.15], c='r', lw=2)

# equal scale x, y
plt.xlim([0, 0.15])
plt.ylim([-0.05, 0.15])

# equal aspect
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel('Ahn kappa0')
plt.ylabel('our kappa0')
plt.title('Ahn vs our kappa0')
plt.show()


In [ ]:
# scatter plot compare kappa0 values
plt.figure()
plt.scatter(df_both['kappa0_ahn'].values, df_both['kappa0_1'].values, c='b', s=5, marker='.')

# plot x=y line
plt.plot([0, 0.15], [0, 0.15], c='r', lw=2)

# equal scale x, y
plt.xlim([0, 0.15])
plt.ylim([-0.05, 0.15])

# equal aspect
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel('Ahn kappa0')
plt.ylabel('our kappa0_1')
plt.title('Ahn vs our kappa0_1 (fc halved)')
plt.show()




# scatter plot compare kappa0 values
plt.figure()
plt.scatter(df_both['kappa0_ahn'].values, df_both['kappa0_2'].values, c='r', s=5, marker='.')

# plot x=y line
plt.plot([0, 0.15], [0, 0.15], c='r', lw=2)

# equal scale x, y
plt.xlim([0, 0.15])
plt.ylim([-0.05, 0.15])

# equal aspect
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel('Ahn kappa0')
plt.ylabel('our kappa0_2')
plt.title('Ahn vs our kappa0_2 (fc doubled)')
plt.show()

In [ ]:
df_both['kappa0_1'].values

In [ ]:
# Make a map view plot of stations colored by the Y0 values
df_sta_calib['y0'] = y0

plt.figure()
plt.scatter(df_sta_calib['slon'].values, df_sta_calib['slat'].values, c=y0, s=60, marker='v', cmap='seismic', edgecolors='k')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.colorbar()
plt.show()

In [ ]:
# ## Compute STN values for spectra

# # find indices where spectra are non-NaN, stack into 2D-array
# # should get equivalent result using either p1 or p2 columns
# print("Computing STN for spectra...", end='')
# t0 = time.time()
# spec_inds = np.where([hasattr(el, '__len__') for el in df['s2'].values ])[0]
# s2 = np.vstack(df['s2'].values[spec_inds], dtype=float)
# s1 = np.vstack(df['s1'].values[spec_inds], dtype=float)

# # compute stn for each row in the 2D-array
# stn = compute_stn(s2, s1, stn_inds)
# df['stn'] = np.nan
# df.loc[spec_inds, 'stn'] = stn
# print(f"done (t = {time.time()-t0:.3f} s)")

# if 'stn' not in dependents: dependents += ['stn']

Visualize the signal-to-noise of the calibration event records

In [ ]:
bins = np.arange(-5, 20, 0.25)
logbins = np.arange(-2, 4, 0.1)
df_calib = df[np.logical_and(df['qmag'] >= calib_mag_range[0], df['qmag'] < calib_mag_range[1])].reset_index(drop=True)

plt.figure()
plt.hist((df_calib['stn']), bins=bins, histtype='step', color='b', label='STN')
plt.axvline(calib_stn_req, color='k', label=f"req. STN = {calib_stn_req:.1f}")
plt.legend()
plt.xlabel("STN")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(np.log10(df_calib['stn']), bins=logbins, histtype='step', color='b', label='STN')
plt.axvline(np.log10(calib_stn_req), color='k', label=f"req. log STN = {np.log10(calib_stn_req):.3f}")
plt.legend()
plt.xlabel("STN")
plt.ylabel("Count")
plt.show()



In [ ]:



# Indices of records that have spectra
spec_inds = np.where([hasattr(el, '__len__') for el in df['s2'].values ])[0]

# Indices of calibration records that have spectra
df_calib_inds = np.where([hasattr(el, '__len__') for el in df_calib['s2'].values ])[0]


# DataFrame containing all records of events greater than the largest 
# calibration event. These are the target events.
# df_target_ev = df_ev[df_ev['qmag'] >= calib_mag_range[1]].reset_index(drop=True)

# target events are all events
df_target_ev = df_ev.copy()

print(f"{len(df_target_ev):,.0f} target events with M >= {np.min(df['qmag']):.2f} ({len(df_target_ev):,.0f} records)")
print(f"{len(np.unique(df_calib['event_id'])):,.0f} calibration events with {calib_mag_range[0]} <= M < {calib_mag_range[1]} ({len(df_calib):,.0f} records)")


df_target_ev['dlogbeta'] = np.nan

# DataFrames with _ev suffix are grouped by event (i.e. one event per row)
# Otherwise, DataFrames have one row per record (station/event combination)

# loop over each target event
for nev in trange(len(df_target_ev), desc="Computing corrected logbeta"):

    # Store the entire row of the target event
    target_ev = df_target_ev.loc[nev]

    # Store a copy of the calibration event records DataFrame
    df_c = df_calib.copy()

    # Get all records in df relating to target event_id
    df_target = df[df['event_id'] == target_ev['event_id']].reset_index(drop=True)

    
    # Filter out calibration event records that:
    #   1) are too shallow or too deep
    #   2) don't share stations with the target event
    #   3) are too low signal-to-noise ratio
    keep_bool = np.all([
        df_c['qdep']>=target_ev['qdep']-calib_zmax, 
        df_c['qdep']<=target_ev['qdep']+calib_zmax,
        np.isin(df_c['stind'].values, target_ev['stind']),
        df_c['stn'] >= calib_stn_req,
        ], axis=0)
    df_c = df_c[keep_bool].reset_index(drop=True)
    
    # rough square filter to avoid computing distances for all calibration events (slightly faster)
    keep_bool = np.all([
        np.abs(df_c['qx']-target_ev['qx']) <= calib_rmax*1000,
        np.abs(df_c['qy']-target_ev['qy']) <= calib_rmax*1000,
    ], axis=0)
    df_c = df_c[keep_bool].reset_index(drop=True)

    # compute station-event distance 
    df_c['tdist'] = np.sqrt((target_ev['qx'] - df_c['qx'].values)**2 + (target_ev['qy'] - df_c['qy'].values)**2)

    # filter out calibration events that are too far from the target event
    df_c = df_c[df_c['tdist'] <= calib_rmax*1000].reset_index(drop=True)

    # simplify df_c by removing unnecessary columns
    df_c = df_c[['stname','stind', 'event_id', 'logbeta']]

    ncalib = len(np.unique(df_c['event_id'].values))

    # only compute dlogbeta if there are enough remaining calibration events
    if ncalib >= ncalib_min:
        df_target_pre = df_target.copy()
        df_target = pd.merge(df_target, df_c, how='inner', on='stind', suffixes=['_t','_c'])
        
        df_target['dlogbeta'] = df_target[f'logbeta_t'] - df_target[f'logbeta_c']
        # print(len(df_target) - np.sum(np.isnan(df_target['dlogbeta'])), len(df_target) - np.sum(np.isnan(df_target['dlogbeta'])))
        if len(df_target) - np.sum(np.isnan(df_target['dlogbeta'])) > ncalib_min:
            df_target_ev.at[nev,'dlogbeta'] = np.nanmedian(df_target['dlogbeta'])

print('loop finished')



In [ ]:
# magnitude_correction = "simple"
# magnitude_correction = "spline"
magnitude_correction = "smoothedspline"

mag_corr_dM = 0.2

df_target_ev = df_target_ev[~np.isnan(df_target_ev['dlogbeta'])].reset_index(drop=True)
df_target_ev['dlogbeta_corr'] = df_target_ev['dlogbeta'].values

# indices where dlogbeta was calculated
not_nan = ~np.isnan(df_target_ev['dlogbeta'])

# Automatically determine the range of magnitude bins
M_range = np.round((
    np.floor(np.min(df_target_ev['qmag'].values) / mag_corr_dM) * mag_corr_dM, 
    np.ceil(np.max(df_target_ev['qmag'].values) / mag_corr_dM) * mag_corr_dM
    ), 4) # fixes rounding errors

# Define the edges and midpoints of the magnitude bins
edges = np.arange(M_range[0], M_range[1]+1*mag_corr_dM, mag_corr_dM)
midpoints = (edges[:-1] + edges[1:])/2

# Assign each target event to a magnitude bin
bin_inds = np.digitize(df_target_ev['qmag'].values, edges) - 1

# Compute the median dlogbeta for each magnitude bin
median_dlbeta = np.empty(len(edges)-1) * np.nan
for i in range(len(median_dlbeta)):
    median_dlbeta[i] = np.nanmedian(df_target_ev['dlogbeta'].values[bin_inds==i])

# Perform simple correction by subtracting the median dlogbeta
if magnitude_correction == "simple":
    for i in range(len(edges)-1):
        df_target_ev.loc[bin_inds==i, 'dlogbeta_corr'] -= median_dlbeta[i]

# Perform spline magnitude correction
elif magnitude_correction == "spline":
    corrections = np.interp(df_target_ev['qmag'].values, midpoints, median_dlbeta)
    df_target_ev['dlogbeta_corr'] = df_target_ev['dlogbeta'].values - corrections
elif magnitude_correction == "smoothedspline":
    from scipy.signal import savgol_filter
    realmids = midpoints[~np.isnan(median_dlbeta)]
    median_dlbeta_smooth = savgol_filter(median_dlbeta[~np.isnan(median_dlbeta)], int(np.floor(len(median_dlbeta)/4)), 3)
    corrections = np.interp(df_target_ev['qmag'].values, realmids, median_dlbeta_smooth)


    # median_dlbeta_smooth = savgol_filter(median_dlbeta, int(np.floor(len(median_dlbeta)/4)), 3)
    # corrections = np.interp(df_target_ev['qmag'].values, midpoints, median_dlbeta_smooth)
    df_target_ev['dlogbeta_corr'] = df_target_ev['dlogbeta'].values - corrections

df_target_ev_out = df_target_ev[['event_id', 'qmag', 'qlon', 'qlat', 'qdep', 'dlogbeta', 'dlogbeta_corr']]

if generate_synthetic:
    np.savetxt(f'/Users/ivandevert/projects/spectral_falloff_ratio/data/beta/betaout_{syn_dataset_name}.txt', df_target_ev_out.values, fmt='%8i %5.2f %15.11f %15.11f %7.3f %15.11f %15.11f')
else:
    np.savetxt(f'/Users/ivandevert/projects/spectral_falloff_ratio/data/beta/betaout_{phase}.txt', df_target_ev_out.values, fmt='%8i %5.2f %15.11f %15.11f %7.3f %15.11f %15.11f')


fig, ax = plt.subplots(2, 1, figsize=(8, 8))
ax[0].scatter(df_target_ev['qmag'], df_target_ev['dlogbeta'], c='gray', s=1, edgecolors='none')
ax[1].scatter(df_target_ev['qmag'], df_target_ev['dlogbeta_corr'], c='gray', s=1, edgecolors='none')
boxplot(df_target_ev['qmag'], df_target_ev['dlogbeta_corr'], xbins=edges, ax=ax[1], color='r')

if magnitude_correction == "simple":
    ax[0].scatter(midpoints, median_dlbeta, c='k', marker='x', label='Bin medians')
    for i in range(len(edges)-1):
        ax[0].plot([edges[i], edges[i+1]], [median_dlbeta[i], median_dlbeta[i]], c='r', lw=1, label='Corrections')
    ax[1].scatter(midpoints, median_dlbeta - median_dlbeta, c='k', marker='x', label='Bin medians')

elif magnitude_correction == "spline":
    xx = np.linspace(M_range[0], M_range[1], 100)
    ax[0].scatter(midpoints, median_dlbeta, c='k', marker='x', label='Knots')
    ax[0].plot(xx, np.interp(xx, midpoints, median_dlbeta), c='r', label='Corrections')

elif magnitude_correction == "smoothedspline":
    xx = np.linspace(M_range[0], M_range[1], 100)
    # ax[0].scatter(midpoints, median_dlbeta, c='k', marker='x', label='Medians')
    # ax[0].scatter(midpoints, median_dlbeta_smooth, c='lime', marker='x', label='Smoothed medians')
    ax[0].scatter(realmids, median_dlbeta_smooth, c='lime', marker='x', label='Smoothed medians')
    # ax[0].plot(xx, np.interp(xx, midpoints, median_dlbeta), c='r', label='Spline')
    # ax[0].plot(xx, np.interp(xx, midpoints, median_dlbeta_smooth), c='g', label='Smoothed spline')
    ax[0].plot(xx, np.interp(xx, realmids, median_dlbeta_smooth), c='g', label='Smoothed spline')



handles, labels = ax[0].get_legend_handles_labels()
unique = [(h, l) for i, (h, l) in enumerate(zip(handles, labels)) if l not in labels[:i]]
ax[0].legend(*zip(*unique), loc='best')
# ax[1].legend()
ax[0].set_xlabel('Magnitude')
ax[1].set_xlabel('Magnitude')
ax[0].set_ylabel(r'$\Delta\log\beta$')
ax[1].set_ylabel(r'$\Delta\log\beta^*$')
ax[0].set_title(r'$\Delta\log\beta$')
ax[1].set_title(r'$\Delta\log\beta^*$')
fig.suptitle(f"{magnitude_correction} magnitude correction")
plt.tight_layout()
plt.show()



In [ ]:
int(np.floor(len(median_dlbeta)/4))

In [ ]:
midpoints[np.isnan(median_dlbeta)]

In [ ]:
median_dlbeta

In [ ]:
midpoints

In [ ]:
xx

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,4))
ax.scatter(df_target_ev['qmag'], df_target_ev['dlogbeta'], c='gray', s=1, edgecolors='none')

if magnitude_correction == "smoothedspline":
    xx = np.linspace(M_range[0], M_range[1], 100)
    # ax[0].scatter(midpoints, median_dlbeta, c='k', marker='x', label='Medians')
    # ax.scatter(midpoints, median_dlbeta_smooth, c='lime', marker='x', label='Smoothed medians')
    # ax[0].plot(xx, np.interp(xx, midpoints, median_dlbeta), c='r', label='Spline')
    # ax.plot(xx, np.interp(xx, midpoints, median_dlbeta_smooth), c='lime', label='Smoothed spline')
    ax.plot(xx, np.interp(xx, realmids, median_dlbeta_smooth), c='lime', label='Smoothed spline')
# ax.scatter(midpoints, median_dlbeta, c='r', marker='x', s=80, label='Bin medians')
ax.scatter(midpoints, median_dlbeta, c='r', marker='x', s=80, label='Bin medians')


handles, labels = ax.get_legend_handles_labels()
unique = [(h, l) for i, (h, l) in enumerate(zip(handles, labels)) if l not in labels[:i]]
ax.legend(*zip(*unique), loc='best')
ax.set_xlabel('Magnitude')
ax.set_ylabel(r'$\Delta\log\beta$')
# ax.set_title(r'$\Delta\log\beta$')
# fig.suptitle(f"{magnitude_correction} magnitude correction")
ax.set_xlim((1,5))
ax.set_ylim((-2, 1))
plt.tight_layout()

if generate_synthetic:
    plt.savefig(paper_figure_dir + f"{phase}_synthetic_magnitude_correction.pdf", bbox_inches='tight')
else:
    plt.savefig(paper_figure_dir + f"{phase}_magnitude_correction.pdf", bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(midpoints, median_dlbeta_smooth-median_dlbeta, c='k', marker='x')
plt.plot(xx, np.interp(xx, midpoints, median_dlbeta_smooth) - np.interp(xx, midpoints, median_dlbeta), c='r')
plt.axhline(0, c='k', ls='--')
plt.xlabel('Magnitude')
plt.show()

In [ ]:
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter


xx = np.linspace(1, 7, 1001)
# cubic spline
s = CubicSpline(midpoints, median_dlbeta)

yy = savgol_filter(median_dlbeta, int(np.floor(len(median_dlbeta)/4)), 3)

plt.figure()
plt.scatter(midpoints, median_dlbeta, c='k', marker='x')
# plt.plot(xx, np.interp(xx, midpoints, median_dlbeta), c='r', label='spline')
# plt.plot(xx, s(xx), c='b', label='cubic spline')
plt.scatter(midpoints, yy, c='g', marker='x',label='savgol')
plt.legend()
# plt.xlim([3, 4.1])
# plt.ylim([-1.4, -0.5])
plt.show()

In [ ]:
# import Rectangle function
from matplotlib.patches import Rectangle


dependents = ['deldist', 's1', 's2', 'logbeta']
st_dep = ['stname', 'slat', 'slon', 'selev', 'sx', 'sy', 'stind']
ev_dep = ['event_id', 'qmag', 'qlon', 'qlat', 'qdep', 'qx',
       'qy', 'dlogbeta', 'dlogbeta_corr']


if generate_synthetic: stind_plot = 0
if not generate_synthetic: 
    if phase == 'p': 
        stind_plot = 58
    elif phase == 's':
        stind_plot = 83

dx = 0.2
edges = np.arange(1.0, 7.3, dx)

df = df_target_ev.explode(st_dep + dependents).reset_index(drop=True, inplace=False)
df_plot = df[df['stind']==stind_plot]

stname = df_plot['stname'].values[0]


df_sta = df.groupby(st_dep, as_index=False)[ev_dep+dependents].agg(list)

keep = [len(el)>50 for el in df_sta['event_id']]
df_sta_good = df_sta[keep]

df_c = df_calib[df_calib['stname']==stname]

fig, axs = plt.subplots(1,3, figsize=[8,3], sharex=True, sharey=True, layout='constrained')
# fig, axs = plt.subplots(1,3, figsize=[8,3], sharex=True, sharey=False, layout='constrained')
axs[0].scatter(df_plot['qmag'], df_plot['logbeta'], c='k', s=1, edgecolors='none', label=r'$\log\beta$')
axs[0].scatter(df_c['qmag'], df_c['logbeta'], c='k', s=1, edgecolors='none')

axs[1].scatter(df_plot['qmag'], df_plot['dlogbeta'], c='k', s=1, edgecolors='none', label=r'$\Delta\log\beta$')
axs[2].scatter(df_plot['qmag'], df_plot['dlogbeta_corr'], c='k', s=1, edgecolors='none', label=r'$\Delta\log\beta^*$')

boxplot(df_plot['qmag'], df_plot['logbeta'], xbins=edges, ax=axs[0], color='r')
boxplot(df_c['qmag'], df_c['logbeta'], xbins=edges, ax=axs[0], color='r')
boxplot(df_plot['qmag'], df_plot['dlogbeta'], xbins=edges, ax=axs[1], color='r')
boxplot(df_plot['qmag'], df_plot['dlogbeta_corr'], xbins=edges, ax=axs[2], color='r')

# shaded rectangle around calibration events range
axs[0].add_patch(Rectangle((calib_mag_range[0], -4.5), calib_mag_range[1]-calib_mag_range[0], 9, fill=True, facecolor='gray', alpha=0.5, edgecolor='none'))

axs[0].set_xlim([1.0, 5])
if phase == 'p': axs[0].set_ylim([-2.0, 1.0])
if phase == 's': axs[0].set_ylim([-2.0, 1.0])
axs[1].set_xlabel('Magnitude')
axs[0].set_ylabel(r'$\log\beta$')
axs[1].set_ylabel(r'$\Delta\log\beta$')
axs[2].set_ylabel(r'$\Delta\log\beta^*$')
axs[1].set_title(f'{stname} (n = {len(df_plot):,.0f})')
# axs[0].legend()
# axs[1].legend()
# axs[2].legend()
axs[0].grid(True)
axs[1].grid(True)
axs[2].grid(True)

# panel labels
for n, axes in enumerate(axs):
    axes.text(0.019, 0.90, string.ascii_lowercase[n], transform=axes.transAxes, 
        size=16)
        
if generate_synthetic:
    plt.savefig(paper_figure_dir + f"{phase}_syn_corrections.pdf", bbox_inches='tight')
else:
    plt.savefig(paper_figure_dir + f"{phase}_corrections.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# plot all qmag vs dlogbeta_corr
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(df_target_ev['qmag'], df_target_ev['dlogbeta_corr'], c='k', s=1, edgecolors='none')
boxplot(df_target_ev['qmag'], df_target_ev['dlogbeta_corr'], xbins=np.arange(1.0, 7.3, 0.1), ax=ax, color='r')

# if generate_synthetic:
#     plt.ylim([-0.05, 0.05])

plt.show()

In [ ]:
for i in np.unique(df['stind']):
    stname = df[df['stind']==i]['stname'].values[0]
    print(i, stname, len(df[df['stind']==i]))